In [15]:
#!/usr/bin/env python
# coding: utf-8


# -*- coding: utf-8 -*-
# Input the output for date and time where ~ is
"""
Updated on Thu Jul 10 10:39:51 2025
"""
import pandas as pd

from datetime import datetime

current_time = datetime.now().strftime('%a %b %d %H:%M:%S %Y')
print(current_time)


#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import getpass
import warnings
import time
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz import process

Thu Jul 10 10:47:19 2025


In [16]:
# ----------------------------------------------------------------------------
# Set up Stata directory (if needed) and define file paths
# ----------------------------------------------------------------------------

# Example: If you or your team members have different local paths
# Input own local path at ~ below using previous users as a reference
username = getpass.getuser()
user_configs = {
    "markd": {
        "AHA-SDC": r"C:\Users\markd\Dropbox\Private Equity Project\ProcessingAnalysis\AHA-SDC-Link",
        "STATA_SYSDIR": r"C:\Program Files\Stata18"
    },
}
if username in user_configs:
    AHA_SDC = user_configs[username]["AHA-SDC"]
    STATA_SYSDIR = user_configs[username]["STATA_SYSDIR"]
else:
    raise ValueError(f"Username '{username}' not recognized. Please add your config.")
# as above, input stata version information including if it is SE or MP (this is a main reason for errors) in the ~ spaces below
import sys
sys.path.append('C:\Program Files\Stata18/utilities')
from pystata import config
config.init('se')
config.status()

# folder names for dropbox may change, any updates should be reflected here
# if working in a subfolder of work_data, implament the subfolder name and your username in the ~ spaces below (username first, then subfolder name)
AP_raw_data_dir   = os.path.join(AHA_SDC, "raw_data")
AP_work_data_dir  = os.path.join(AHA_SDC, "work_data")
AP_final_data_dir = os.path.join(AHA_SDC, "final_data")
os.makedirs(AP_raw_data_dir,   exist_ok=True)
os.makedirs(AP_work_data_dir,  exist_ok=True)
os.makedirs(AP_final_data_dir, exist_ok=True)

# Input file names below for your desired match keys in Pitchbook and AHA data
# Filenames for AHA data is likely to be "aha_match_keys.dta" 
# Rember to input information in all ~ spaces below and above
# Filenames for Pitchbook data is likely to be similar to "pitchbook_match_keys... .dta"

# Filenames for Fuzzy Match
Preqin_file = os.path.join(AP_work_data_dir, "sdc_match_keys.dta")
AHA_file   = os.path.join(AP_work_data_dir, "aha_match_keys.dta")

    System information
      Python version         3.13.0
      Stata version          Stata 18.0 (SE)
      Stata library path     C:\Program Files\Stata18\se-64.dll
      Stata initialized      True
      sfi initialized        True

    Settings
      graphic display        True
      graphic size           width = default, height = default
      graphic format         svg


<>:21: SyntaxWarning: invalid escape sequence '\P'
<>:21: SyntaxWarning: invalid escape sequence '\P'
C:\Users\markd\AppData\Local\Temp\ipykernel_12152\2165547053.py:21: SyntaxWarning: invalid escape sequence '\P'
  sys.path.append('C:\Program Files\Stata18/utilities')


In [17]:
import time
from rapidfuzz import fuzz, process
import pandas as pd
import re

def fuzzy_match_verify(
    id_a,
    id_b,
    id_list_a,
    id_list_b,
    unique_id_list_b,
    name_trim_list_a,
    name_trim_list_b,
    name_raw_list_a,
    name_raw_list_b,
    city_list_b,
    city_list_a,
    state_list_a,
    state_list_b,
    zip5_list_b,
    zip5_list_a,
    address_list_a,
    address_list_b,
    level_list_b,
    match_threshold=80,
    match_scorer=fuzz.token_set_ratio,
    verify_threshold=80,
    verify_scorer=fuzz.ratio,
    choice_limit=5
):
    """
    All-to-all fuzzy match with second-pass verification and metadata retention.
    """

    # First-pass match results
    id_matched_list_a = []
    id_matched_list_b = []
    unique_id_matched_list_b = []
    name_trim_matched_list_a = []
    name_trim_matched_list_b = []
    name_raw_matched_list_a = []
    name_raw_matched_list_b = []
    city_matched_list_b = []
    city_matched_list_a = []
    state_matched_list_a = []
    state_matched_list_b = []
    zip5_matched_list_b = []
    zip5_matched_list_a = []
    address_matched_list_b = []
    address_matched_list_a = []
    
    level_matched_list_b = []
    match_score_list = []

    for i in range(len(id_list_a)):
        query = name_trim_list_a[i]
        if not isinstance(query, str) or not query.strip():
            continue

        match_list = process.extract(
            query,
            name_trim_list_b,
            limit=choice_limit,
            scorer=match_scorer
        )

        for k in range(len(match_list)):
            b_match_val = match_list[k][0]
            match_score = match_list[k][1]
            b_index = name_trim_list_b.index(b_match_val)  # safe due to exact match

            if match_score < match_threshold:
                continue

            id_matched_list_a.append(id_list_a[i])
            id_matched_list_b.append(id_list_b[b_index])
            unique_id_matched_list_b.append(unique_id_list_b[b_index])
            name_trim_matched_list_a.append(name_trim_list_a[i])
            name_trim_matched_list_b.append(name_trim_list_b[b_index])
            name_raw_matched_list_a.append(name_raw_list_a[i])
            name_raw_matched_list_b.append(name_raw_list_b[b_index])
            city_matched_list_b.append(city_list_b[b_index])
            city_matched_list_a.append(city_list_a[i])
            state_matched_list_a.append(state_list_a[i])
            state_matched_list_b.append(state_list_b[b_index])
            zip5_matched_list_b.append(zip5_list_b[b_index])
            zip5_matched_list_a.append(zip5_list_a[i])
            address_matched_list_b.append(address_list_b[b_index])
            address_matched_list_a.append(address_list_a[i])
            level_matched_list_b.append(level_list_b[b_index])
            match_score_list.append(match_score)

    # Second-pass verification
    id_verified_list_a = []
    id_verified_list_b = []
    unique_id_verified_list_b = []
    name_trim_verified_list_a = []
    name_trim_verified_list_b = []
    name_raw_verified_list_a = []
    name_raw_verified_list_b = []
    city_verified_list_b = []
    city_verified_list_a = []
    state_verified_list_a = []
    state_verified_list_b = []
    zip5_verified_list_b = []
    zip5_verified_list_a = []
    address_verified_list_a = []
    address_verified_list_b = []
    level_verified_list_b = []
    match_score_verified_list = []

    start_time = time.perf_counter()
    for i in range(len(id_matched_list_a)):
        a_clean = name_trim_matched_list_a[i].strip().lower()
        b_clean = name_trim_matched_list_b[i].strip().lower()
        score = verify_scorer(a_clean, b_clean)

        if score >= verify_threshold:
            id_verified_list_a.append(id_matched_list_a[i])
            id_verified_list_b.append(id_matched_list_b[i])
            unique_id_verified_list_b.append(unique_id_matched_list_b[i])
            name_trim_verified_list_a.append(name_trim_matched_list_a[i])
            name_trim_verified_list_b.append(name_trim_matched_list_b[i])
            name_raw_verified_list_a.append(name_raw_matched_list_a[i])
            name_raw_verified_list_b.append(name_raw_matched_list_b[i])
            city_verified_list_b.append(city_matched_list_b[i])
            city_verified_list_a.append(city_matched_list_a[i])
            state_verified_list_a.append(state_matched_list_a[i])
            state_verified_list_b.append(state_matched_list_b[i])
            zip5_verified_list_b.append(zip5_matched_list_b[i])
            zip5_verified_list_a.append(zip5_matched_list_a[i])
            address_verified_list_a.append(address_matched_list_a[i])
            address_verified_list_b.append(address_matched_list_b[i])
            level_verified_list_b.append(level_matched_list_b[i])
            match_score_verified_list.append(score)

    end_time = time.perf_counter()
    print(f"Verify Process — {len(id_verified_list_a)} matches found in {end_time - start_time:.2f}s")

    # Build final DataFrame
    match_dict = {
        f"{id_a}": id_verified_list_a,
        f"{id_b}": id_verified_list_b,
        "unique_id_b": unique_id_verified_list_b,
        "name_trim_a": name_trim_verified_list_a,
        "name_trim_b": name_trim_verified_list_b,
        "name_raw_a": name_raw_verified_list_a,
        "name_raw_b": name_raw_verified_list_b,
        "city_b": city_verified_list_b,
        "city_a": city_verified_list_a,
        "state_a": state_verified_list_a,
        "state_b": state_verified_list_b,
        "level_b": level_verified_list_b,
        "zip5_b": zip5_verified_list_b,
        "zip5_a": zip5_verified_list_a,
        "address_a": address_verified_list_a,
        "address_b": address_verified_list_b,
        "match_score": match_score_verified_list
    }
    for k, v in match_dict.items():
        print(f"{k}: {len(v)}")
    
    return pd.DataFrame(match_dict)

In [18]:
# ----------------------------------------
# Helper: Clean hospital or firm name
# ----------------------------------------
def clean_name(name):
    if not isinstance(name, str):
        return ""
    
    name = name.lower().strip()
    name = re.sub(r"\s+", " ", name)

    # Remove corporate suffixes
    for term in [" inc", " llc", " corp", " corporation", " ltd", " limited"]:
        name = name.replace(term, "")

    # Remove punctuation
    name = re.sub(r"[^a-z0-9 ]", "", name)

    # Normalize common abbreviations
    name = re.sub(r"\bst\b", "saint", name)
    name = re.sub(r"\bmt\b", "mount", name)

    # Remove generic stopwords
    stopwords = {
        "hospital", "center", "medical", "health", "the",
        "care", "healthcare", "system", "group", "services",
        "inc", "llc", "corp", "corporation", "ltd", "limited",
        "regional", "holdings", "systems", "hosp", "ctr", "mem", "memorial", "reg"
    }
    name = " ".join([w for w in name.split() if w not in stopwords])
    return re.sub(r"\s+", " ", name).strip()

# ----------------------------------------------------------------------------
# Main function

def main():
    print("=== Loading 'pe_acquisition' from:", Preqin_file)
    acquisition_df = pd.read_stata(Preqin_file)
    acquisition_df = acquisition_df[acquisition_df['name_trim'].notna()]
    
    # Clean PE names
    acquisition_df["rawname_pe"] = acquisition_df["name_raw"]
    acquisition_df["trimname_pe"] = acquisition_df["name_trim"].fillna("").astype(str).apply(clean_name)

    # Extract PE columns
    pe_id     = acquisition_df['id'].tolist()
    pe_raw    = acquisition_df['rawname_pe'].fillna("").tolist()
    pe_clean  = acquisition_df["trimname_pe"].tolist()
    pe_state  = acquisition_df["state"].fillna("").tolist()
    pe_city   = acquisition_df["city"].fillna("").tolist()
    pe_zip5   = acquisition_df["zip"].fillna("").tolist()
    pe_address = acquisition_df["address"].fillna("").tolist()

    print("=== Loading 'hospital' from:", AHA_file)
    hospital_df = pd.read_stata(AHA_file)
    hospital_df = hospital_df[hospital_df['name_trim'].notna()]
    
    # Clean hospital names
    hospital_df["rawname_aha"] = hospital_df["name_raw"]
    hospital_df["trimname_aha"] = hospital_df["name_trim"].fillna("").astype(str).apply(clean_name)

    # Extract hospital columns
    hosp_id     = hospital_df['id'].tolist()
    hosp_unique_id = hospital_df['aha_unique_id'].tolist()
    hosp_raw    = hospital_df['rawname_aha'].fillna("").tolist()
    hosp_clean  = hospital_df["trimname_aha"].tolist()
    hosp_city   = hospital_df["city"].fillna("").tolist()
    hosp_state  = hospital_df["state"].fillna("").tolist()
    hosp_zip5   = hospital_df["zip5"].fillna("").tolist()
    hosp_address = hospital_df["addr"].fillna("").tolist()
    hosp_level = hospital_df["level"].astype(str).fillna("").tolist()
    
    print("\n=== Fuzzy matching: PE acquisitions to hospital names ===")
    start_time = time.time()

    direct_candidates = fuzzy_match_verify(
        id_a = "id_a",
        id_b = "id_b",

        id_list_a = pe_id,
        id_list_b = hosp_id,

        unique_id_list_b = hosp_unique_id,

        name_trim_list_a = pe_clean,
        name_trim_list_b = hosp_clean,

        name_raw_list_a = pe_raw,
        name_raw_list_b = hosp_raw,

        city_list_b = hosp_city,
        city_list_a = pe_city,

        state_list_a = pe_state,
        state_list_b = hosp_state,

        zip5_list_b = hosp_zip5,
        zip5_list_a = pe_zip5,

        address_list_b = hosp_address,
        address_list_a = pe_address,

        level_list_b = hosp_level,

        # Fuzzy match parameters

        match_threshold  = 75,
        verify_threshold = 75,
        match_scorer     = fuzz.token_set_ratio,
        verify_scorer    = fuzz.ratio,
        choice_limit     = 10
    )
    end_time = time.time()
    print(f"Fuzzy match found {len(direct_candidates)} matched pairs. Time = {end_time - start_time:.2f}s")

    # Save to CSV
    # Input username in ~ below then input your desired output file name in the following ~
    # As a convention, the output file name should be of the form "Python-AHA-Pitchbook-Link-PB Sample Name (COP,HIP, or COP+HIP)-Keywording type (exclusive, inclusive, or exclusive+inclusive).csv"
    direct_file = os.path.join(AP_work_data_dir, "Python-AHA-SDC-Link.csv")
    direct_candidates.to_csv(direct_file, index=False)
    print("Direct hospital/system fuzzy matches saved to:", direct_file)

if __name__ == "__main__":
    main()

=== Loading 'pe_acquisition' from: C:\Users\markd\Dropbox\Private Equity Project\ProcessingAnalysis\AHA-SDC-Link\work_data\sdc_match_keys.dta
=== Loading 'hospital' from: C:\Users\markd\Dropbox\Private Equity Project\ProcessingAnalysis\AHA-SDC-Link\work_data\aha_match_keys.dta


C:\Users\markd\AppData\Local\Temp\ipykernel_12152\1799194159.py:54: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  hospital_df = pd.read_stata(AHA_file)



=== Fuzzy matching: PE acquisitions to hospital names ===
Verify Process — 5328 matches found in 0.03s
id_a: 5328
id_b: 5328
unique_id_b: 5328
name_trim_a: 5328
name_trim_b: 5328
name_raw_a: 5328
name_raw_b: 5328
city_b: 5328
city_a: 5328
state_a: 5328
state_b: 5328
level_b: 5328
zip5_b: 5328
zip5_a: 5328
address_a: 5328
address_b: 5328
match_score: 5328
Fuzzy match found 5328 matched pairs. Time = 182.90s
Direct hospital/system fuzzy matches saved to: C:\Users\markd\Dropbox\Private Equity Project\ProcessingAnalysis\AHA-SDC-Link\work_data\Python-AHA-SDC-Link.csv
